### **Corrida local opcional con RTX 4080**

Este cuaderno propone una ruta compacta para ejecutar una **corrida local opcional** con un modelo real de captioning y, si se desea, una demostración pequeña de generación texto->imagen.

La lógica metodológica se mantiene: **recuperación + evidencia + generación grounded**. La GPU se usa como una extensión experimental, no como sustituto de la evidencia estructurada.


#### 1. Objetivo

- cargar la muestra real pequeña ya normalizada,
- identificar qué registros tienen activos descargables o locales,
- ejecutar captioning real con BLIP cuando exista imagen,
- comparar la salida del modelo con la salida grounded del proyecto,
- dejar preparada una ruta opcional y acotada para generación texto->imagen.


In [1]:
# Detecta la raíz del proyecto y cargar los registros normalizados.
from pathlib import Path
import json
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
BASE_DIR = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_DIR = BASE_DIR / "data_processed"
RECORDS_PATH = DATA_DIR / "records_master.jsonl"


def load_jsonl(path: Path):
    # Leer un archivo JSONL simple en memoria.
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

records = load_jsonl(RECORDS_PATH)
df = pd.DataFrame(records)
print("Directorio base:", BASE_DIR)
print("Registros cargados:", len(df))
df[["record_id", "source", "modality_type", "title", "split"]]


Directorio base: /workspace/Semana6/Proyecto/Patrimonio_Andino_Grounded
Registros cargados: 7


,record_id,source,modality_type,title,split
0,okr_kh_0068_view_01,open_khipu,no_image,"Khipu KH0068 (UR1057, AS057)",train
1,okr_kh_0082_view_01,open_khipu,no_image,Khipu KH0082 (AS069),train
2,okr_kh_0323_view_01,open_khipu,no_image,Khipu KH0323 (UR087),dev
3,okr_kh_0328_view_01,open_khipu,no_image,Khipu KH0328 (UR092),train
4,par_1932_01_0005_photo_01,paracas,photo,Fragmento textil Paracas 1932.01.0005,train
5,par_1935_32_0212_photo_01,paracas,photo,Fragmentos de manto Paracas 1935.32.0212,test
6,par_xrf_em1932_01_0013e,paracas,xrf_map,Mapa XRF de muestra textil Paracas EM1932.01.0...,challenge


In [2]:
# Verifica si la GPU está disponible antes de ejecutar una corrida opcional.
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA disponible:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Dispositivos CUDA:", torch.cuda.device_count())
        print("Nombre GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("No fue posible verificar CUDA:", exc)


PyTorch: 2.4.1+cu121
CUDA disponible: True
Dispositivos CUDA: 1
Nombre GPU: NVIDIA GeForce RTX 4080 SUPER


#### **2. Selección de registros con activos**

En esta carpeta no se redistribuyen imágenes patrimoniales externas. Por eso, esta sección sirve para verificar si ya existen activos locales o URLs utilizables para una corrida real.


In [3]:
def has_asset(row):
    image_path = row.get("image_path")
    image_url = row.get("image_url")
    return bool(image_path) or bool(image_url)

df_assets = df[df.apply(has_asset, axis=1)].copy()
print("Registros con algún activo utilizable:", len(df_assets))
df_assets[["record_id", "modality_type", "title", "image_path", "image_url"]]


Registros con algún activo utilizable: 0


,record_id,modality_type,title,image_path,image_url


#### **3. Captioning real opcional con BLIP**

La celda siguiente está desactivada por defecto. Se activa solo si ya existen imágenes locales válidas y si el entorno tiene `transformers`, `torch` y acceso al checkpoint.


In [4]:
ENABLE_REAL_CAPTIONING = False

if ENABLE_REAL_CAPTIONING:
    import torch
    from PIL import Image
    from transformers import BlipProcessor, BlipForConditionalGeneration

    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    model = BlipForConditionalGeneration.from_pretrained(
        "Salesforce/blip-image-captioning-base"
    ).to(device)

    real_outputs = []
    for row in df_assets.to_dict(orient="records"):
        image_path = row.get("image_path")
        if not image_path:
            continue
        path = (BASE_DIR / image_path).resolve()
        if not path.exists():
            continue
        image = Image.open(path).convert("RGB")
        inputs = processor(images=image, return_tensors="pt").to(device)
        generated_ids = model.generate(**inputs, max_new_tokens=35)
        caption = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        real_outputs.append({
            "record_id": row["record_id"],
            "title": row["title"],
            "caption_blip": caption,
        })

    df_real = pd.DataFrame(real_outputs)
    print("Captions reales generados:", len(df_real))
    display(df_real)
else:
    print("ENABLE_REAL_CAPTIONING = False. Activa esta celda solo si ya tienes imágenes locales y modelos instalados.")


ENABLE_REAL_CAPTIONING = False. Activa esta celda solo si ya tienes imágenes locales y modelos instalados.


#### 4. Comparación con la salida grounded

Aunque el modelo real produzca un caption razonable, la comparación relevante es con la salida grounded del proyecto, porque allí intervienen estructura, metadatos y recuperación.


In [5]:
outputs_dir = BASE_DIR / "outputs" / "captions"
grounded_rows = []
for path in sorted(outputs_dir.glob("*.json")):
    obj = json.loads(path.read_text(encoding="utf-8"))
    grounded_rows.append({
        "record_id": obj.get("record_id", path.stem),
        "caption_grounded": obj.get("caption_factual") or obj.get("caption") or ""
    })

df_grounded = pd.DataFrame(grounded_rows)
print("Salidas grounded disponibles:", len(df_grounded))
df_grounded.head()


Salidas grounded disponibles: 7


,record_id,caption_grounded
0,okr_kh_0068_view_01,"Khipu con 589 cordeles, 11 colores registrados..."
1,okr_kh_0082_view_01,"Khipu con 1831 cordeles, 46 colores registrado..."
2,okr_kh_0323_view_01,"Khipu con 370 cordeles, 23 colores registrados..."
3,okr_kh_0328_view_01,"Khipu con 61 cordeles, 7 colores registrados y..."
4,par_1932_01_0005_photo_01,Textile de la cultura Paracas con descripción ...


#### 5. Generación texto->imagen opcional y acotada

Esta parte debe tratarse como una demostración técnica, no como reemplazo del análisis patrimonial. La recomendación es usarla solo con prompts descriptivos muy controlados y en un subconjunto mínimo.


In [6]:
ENABLE_TEXT_TO_IMAGE = False

if ENABLE_TEXT_TO_IMAGE:
    import torch
    from diffusers import DiffusionPipeline

    device = "cuda" if torch.cuda.is_available() else "cpu"
    pipe = DiffusionPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        use_safetensors=True,
    )
    pipe = pipe.to(device)

    prompt = "A museum documentation photo of an Andean knotted cord assembly on a neutral background"
    image = pipe(prompt, num_inference_steps=25).images[0]
    output_path = BASE_DIR / "outputs" / "figuras" / "demo_sdxl_optional.png"
    image.save(output_path)
    print("Imagen guardada en:", output_path)
else:
    print("ENABLE_TEXT_TO_IMAGE = False. Déjalo desactivado para la ruta base del proyecto.")


ENABLE_TEXT_TO_IMAGE = False. Déjalo desactivado para la ruta base del proyecto.


#### 6. Lectura metodológica

- El captioning real puede aportar una línea descriptiva adicional.
- La salida grounded sigue siendo preferible cuando el registro depende de estructura o contexto.
- En Open Khipu, la ausencia de imagen no invalida el análisis, obliga a explicitar la dependencia estructural.
- En Paracas XRF, una imagen técnica pide una nota técnica y no un caption patrimonial estándar.
